<a href="https://colab.research.google.com/github/ereznaamansapienza/mnlp/blob/banana/mnlp2_banana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

## Config

In [1]:
import torch
import gc
import os
import json
import numpy as np
import pandas as pd
import random
import requests

from datasets import load_dataset, Dataset, load_from_disk

from transformers import AutoModelForCausalLM, AutoTokenizer

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from typing import Dict, List, Tuple, Callable, Optional
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

from tqdm.auto import tqdm

import re
import string

from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
import nltk

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download('punkt_tab')

from typing_extensions import override

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

ds_url = "sapienzanlp-course-materials/hw-mnlp-2026"
output_dir = "mnlp/hw_2"
blind_ranking_path = "https://raw.githubusercontent.com/ereznaamansapienza/mnlp/refs/heads/master/doubleN-blind-e5-mnrl-reranker-minilm-ft-full.jsonl"
test_ranking_path = "https://raw.githubusercontent.com/ereznaamansapienza/mnlp/refs/heads/master/doubleN-test-e5-mnrl-reranker-minilm-ft-full.jsonl"

cuda


In [3]:
SEED = 42

def set_seed(seed: int = 42):
  random.seed(seed)
  np.random.seed(seed)

  torch.manual_seed(seed)

  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

set_seed(SEED)

## DataModule

In [4]:
class DataModule:
    def __init__(self, dev_size: float = 0.2, seed: int = 42):
        self.ds = None
        self.dev_size = dev_size
        self.seed = seed

    # Loading
    def load(self, url: str):
        self.ds = load_dataset(url)
        return self.ds

    def to_dataframe(self, split_name: str) -> pd.DataFrame:
        return pd.DataFrame(self.ds[split_name])

    # Splits
    def build_train_val_split(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        train_df_full = self.to_dataframe("train")
        train_df, val_df = train_test_split(
            train_df_full,
            test_size=self.dev_size,
            random_state=self.seed,
            shuffle=True,
        )
        return train_df, val_df

    def build_cv_splits(
        self,
        n_folds: int = 5,
        split_name: str = "train",
        stratify_col: str = "wikipedia_title",
    ) -> List[Tuple[pd.DataFrame, pd.DataFrame]]:
        df = self.to_dataframe(split_name).reset_index(drop=True)

        if stratify_col is not None:
            kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=self.seed)
            split_iter = kf.split(df, df[stratify_col])
        else:
            kf = KFold(n_splits=n_folds, shuffle=True, random_state=self.seed)
            split_iter = kf.split(df)

        folds = []
        for train_idx, val_idx in split_iter:
            folds.append((
                df.iloc[train_idx].reset_index(drop=True),
                df.iloc[val_idx].reset_index(drop=True),
            ))
        return folds

    # Dataset builders
    def build_full_train_dataset(
        self,
        dataset_factory: Callable,
        **factory_kwargs,
    ) -> Tuple[pd.DataFrame, Dataset]:
        """Builds a dataset from the entire training split with no val holdout."""
        train_df = self.to_dataframe("train")
        dataset = dataset_factory(train_df, **factory_kwargs)
        return train_df, dataset

    def build_train_val_datasets(
        self,
        dataset_factory: Callable,
        **factory_kwargs,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, Dataset, Dataset]:
        """Splits train into train/val and builds datasets for both."""
        train_df, val_df = self.build_train_val_split()
        train_dataset = dataset_factory(train_df, **factory_kwargs)
        val_dataset = dataset_factory(val_df,   **factory_kwargs)
        return train_df, val_df, train_dataset, val_dataset


## LM Generators

In [5]:
class LMGenerator():
  def __init__(self, model_name: str, max_new_tokens: int = 64, supports_thinking=False):
      self.model_name = model_name
      self.model = self.load_model()
      self.max_new_tokens = max_new_tokens
      self.supports_thinking = supports_thinking
      self.tokenizer = self.load_tokenizer()

      if self.tokenizer.pad_token is None:
        self.tokenizer.pad_token = self.tokenizer.eos_token

      # For decoder-only batched generation
      self.tokenizer.padding_side = "left"

  def load_model(self):
      return AutoModelForCausalLM.from_pretrained(
          self.model_name,
          torch_dtype="auto",
          device_map="auto"
      )

  def load_tokenizer(self):
      return AutoTokenizer.from_pretrained(self.model_name)

  def build_model_input(self, prompt: str, thinking: bool = False) -> str:
      messages = [
            {"role": "user", "content": prompt}
        ]

      kwargs = {
          "tokenize": False,
          "add_generation_prompt": True,
      }

      if self.supports_thinking:
          kwargs["enable_thinking"] = thinking

      return self.tokenizer.apply_chat_template(
          messages,
          **kwargs,
      )

  def clean_answer(self, text: str) -> str:
        text = text.strip()

        if "</think>" in text:
            text = text.split("</think>")[-1].strip()

        text = text.split("\n")[0].strip()

        for prefix in ["Answer:", "The answer is:", "The answer is"]:
            if text.lower().startswith(prefix.lower()):
                text = text[len(prefix):].strip()

        return text.strip(" .")

  def generate_answers_batch(
      self,
      prompts: List[str],
      thinking: bool = False,
    ) -> List[Tuple[str, str]]:
    texts = [
        self.build_model_input(prompt, thinking=thinking)
        for prompt in prompts
    ]

    model_inputs = self.tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        add_special_tokens=False,
    ).to(self.model.device)

    with torch.inference_mode():
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
        )

    input_length = model_inputs["input_ids"].shape[1]
    output_ids = generated_ids[:, input_length:]

    decoded_outputs = self.tokenizer.batch_decode(
        output_ids,
        skip_special_tokens=True,
    )

    results = []
    for text in decoded_outputs:
        answer = self.clean_answer(text)
        thinking_content = ""
        results.append((answer, thinking_content))

    return results

  def generate_answer(self, prompt: str, thinking: bool = False):
      return self.generate_answers_batch([prompt], thinking=thinking)[0]

In [6]:
class QALLM(LMGenerator):
    def __init__(
        self,
        model_name: str = "Qwen/Qwen3-0.6B",
        max_new_tokens: int = 64,
    ):
        super().__init__(
            model_name=model_name,
            max_new_tokens=max_new_tokens,
            supports_thinking=True,
        )

In [7]:
class SmolLM(LMGenerator):
    def __init__(
        self,
        model_name: str = "HuggingFaceTB/SmolLM2-1.7B-Instruct",
        max_new_tokens: int = 64,
    ):
        super().__init__(
            model_name=model_name,
            max_new_tokens=max_new_tokens,
            supports_thinking=False,
        )

## RetrievalStore

In [8]:
class RetrievalStore:
    def __init__(self, top_k: int = 3):
      self.top_k = top_k

    def load_rankings_jsonl(self, path: str) -> Dict[str, List[int]]:
        rankings = {}

        # URL case
        if path.startswith("http://") or path.startswith("https://"):
            response = requests.get(path)
            response.raise_for_status()

            lines = response.text.splitlines()

        # Local file case
        else:
            with open(path, "r", encoding="utf-8") as f:
                lines = f.readlines()

        for line_no, line in enumerate(lines, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)

                query_id, ranking = next(iter(obj.items()))

                rankings[str(query_id)] = [int(i) for i in ranking]

            except json.JSONDecodeError as e:
                print(f"Skipping invalid JSON on line {line_no}: {e}")

        return rankings

    def attach_rankings_from_jsonl(self, df: pd.DataFrame, path: str) -> pd.DataFrame:
        df = df.copy()
        rankings = self.load_rankings_jsonl(path)

        # Series.map accepts a dict-like mapping and maps values by key
        df["ranking"] = df["query_id"].astype(str).map(rankings)

        missing = df[df["ranking"].isna()]["query_id"].tolist()

        if missing:
            raise ValueError(
                f"Missing rankings for {len(missing)} query_ids. "
                f"First missing examples: {missing[:5]}"
            )

        df["retrieved_chunks"] = df["ranking"].apply(
            lambda ranking: list(ranking[:self.top_k])
        )

        self._validate_indices(df)

        return df

    def _validate_indices(self, df: pd.DataFrame) -> None:
        for row in df.itertuples():
            n_candidates = len(row.candidate_chunks)

            for idx in row.retrieved_chunks:
                if idx < 0 or idx >= n_candidates:
                    raise ValueError(
                        f"Invalid chunk index {idx} for query_id={row.query_id}. "
                        f"n_candidates={n_candidates}"
                    )

    def get_rag_indices(self, row) -> List[int]:
      return list(row.retrieved_chunks[:self.top_k])

    def get_oracle_indices(self, row) -> List[int]:
      retrieved = list(row.retrieved_chunks[:self.top_k])
      correct_idx = int(row.answer_pos)

      if correct_idx in retrieved:
          retrieved.remove(correct_idx)
      else:
          retrieved = retrieved[:-1]

      return [correct_idx] + retrieved


## PromptBuilder

In [9]:
class PromptBuilder:
  def __init__(self, intro="Given the following information:", command="Reply to this question:"):
    self.intro = intro
    self.command = command

  def build_baseline_prompt(self, query: str) -> str:
    return f"""
      Reply to this question in the shortest possible valid way, do not explain the answer:
      {query}
      """

  def build_zero_shot_prompt(self, query: str, chunks: List[str]) -> str:
    numbered_chunks = "\n".join(
          f"{i + 1}. {chunk}" for i, chunk in enumerate(chunks)
      )

    return self.intro + "\n" + numbered_chunks + "\n" + self.command + "\n" + query

In [10]:
evaluation_prompt = """
You are evaluating a generated answer for a question-answering task.

Question:
{query}

Gold short answer:
{short_answer}

Generated answer:
{generated_answer}

Evaluation criterion:
Return 1 if the generated answer contains the gold short answer in any valid form.
Valid forms include paraphrases, spelling variants, number-word equivalence, or a longer sentence that clearly contains the correct answer.
Return 0 if the generated answer does not contain the correct answer, contradicts it, gives a different entity/date/place/person, or is too vague.

Important:
- Do not reward an answer only because it is fluent.
- Do not use external knowledge.
- Judge only whether the generated answer expresses the gold short answer.
- Output only valid JSON.

Output format:
{"score": 0}
or
{"score": 1}
"""

## AnswerEvaluator

In [11]:
class AnswerEvaluator:

    @staticmethod
    def normalize_text(text: str) -> str:
        text = text.lower()
        text = text.translate(
            str.maketrans("", "", string.punctuation)
        )
        text = re.sub(r"\s+", " ", text).strip()
        return text

    @staticmethod
    def _flatten_short_answer(short_answer):
        """
        Handles cases like:
        - ["answer"]
        - "answer"
        - ["a", "b"] → "a b"
        """
        if isinstance(short_answer, list):
            return " ".join(str(x) for x in short_answer)
        return str(short_answer)

    @staticmethod
    def get_EM(generated_answer: str, short_answer) -> int:
        generated_answer = AnswerEvaluator.normalize_text(generated_answer)
        short_answer = AnswerEvaluator.normalize_text(
            AnswerEvaluator._flatten_short_answer(short_answer)
        )
        return int(generated_answer == short_answer)

    @staticmethod
    def get_subEM(generated_answer: str, short_answer) -> int:
        generated_answer = AnswerEvaluator.normalize_text(generated_answer)
        short_answer = AnswerEvaluator.normalize_text(
            AnswerEvaluator._flatten_short_answer(short_answer)
        )
        return int(
            generated_answer in short_answer
            or short_answer in generated_answer
        )

    @staticmethod
    def get_METEOR(generated_answer: str, short_answer) -> float:
        short_answer = AnswerEvaluator._flatten_short_answer(short_answer)

        generated_tokens = word_tokenize(generated_answer)
        short_tokens = word_tokenize(short_answer)

        return meteor_score([short_tokens], generated_tokens)

    @staticmethod
    def evaluate_example(example: Dict) -> Dict:
        """
        Expects:
        {
            "generated_answer": str,
            "short_answer": str | list[str]
        }
        """

        generated_answer = example["generated_answer"]
        short_answer = example["short_answer"]

        return {
            "EM": AnswerEvaluator.get_EM(generated_answer, short_answer),
            "subEM": AnswerEvaluator.get_subEM(generated_answer, short_answer),
            "METEOR": AnswerEvaluator.get_METEOR(generated_answer, short_answer),
        }

    @staticmethod
    def evaluate_dataset(dataset: List[Dict]) -> Dict:

        em_scores = []
        subem_scores = []
        meteor_scores = []

        for example in dataset:

            result = AnswerEvaluator.evaluate_example(example)

            em_scores.append(result["EM"])
            subem_scores.append(result["subEM"])
            meteor_scores.append(result["METEOR"])

        n = len(dataset)

        return {
            "EM": sum(em_scores) / n if n else 0,
            "subEM": sum(subem_scores) / n if n else 0,
            "METEOR": sum(meteor_scores) / n if n else 0,
        }

## LLMJudge

## AgreementAnalyzer

## ExperimentRunner

In [12]:
class ExperimentRunner:
  def __init__(
        self,
        prompt_builder,
        retrieval_store,
        top_k: int = 3,
        seed: int = 42,
    ):
    self.prompt_builder = prompt_builder
    self.retrieval_store = retrieval_store
    self.top_k = top_k

  def build_prompt_from_row(self, row, setting: str) -> Tuple[str, List[int]]:
    if setting == "baseline":
        prompt = self.prompt_builder.build_baseline_prompt(row.query)
        return prompt, []

    if setting == "rag":
        chunk_indices = self.retrieval_store.get_rag_indices(row)
        chunks = [row.candidate_chunks[i] for i in chunk_indices]
        prompt = self.prompt_builder.build_zero_shot_prompt(row.query, chunks)
        return prompt, chunk_indices

    if setting == "oracle":
        chunk_indices = self.retrieval_store.get_oracle_indices(row)
        chunks = [row.candidate_chunks[i] for i in chunk_indices]
        prompt = self.prompt_builder.build_zero_shot_prompt(row.query, chunks)
        return prompt, chunk_indices

  def retrieve_answers(
      self,
      df: pd.DataFrame,
      model,
      setting: str,
      thinking: bool = False,
      batch_size: int = 8,
    ):

    df = df.to_pandas() if not isinstance(df, pd.DataFrame) else df

    prompts = []
    metadata = []

    # Build prompts and metadata
    for row in df.itertuples():
        prompt, chunk_indices = self.build_prompt_from_row(row, setting)

        prompts.append(prompt)

        metadata.append({
            "query_id": row.query_id,
            "retrieved_chunks": chunk_indices,
            "augmented_prompt": prompt,
            "short_answer": getattr(row, "short_answer", None),
            "answer_pos": getattr(row, "answer_pos", None),
            "query": row.query,
        })

    results = []

    # Generate in batches
    for start in tqdm(
        range(0, len(prompts), batch_size),
        total=(len(prompts) + batch_size - 1) // batch_size,
        desc=f"{setting} generation",
    ):
        end = start + batch_size

        batch_prompts = prompts[start:end]
        batch_metadata = metadata[start:end]

        batch_answers = model.generate_answers_batch(
            batch_prompts,
            thinking=thinking,
        )

        for meta, (answer, thinking_content) in zip(batch_metadata, batch_answers):
            row_result = {
                "query_id": meta["query_id"],
                "retrieved_chunks": meta["retrieved_chunks"],
                "augmented_prompt": meta["augmented_prompt"],
                "generated_answer": answer,
            }

            # Fields for evaluation
            row_result["short_answer"] = meta["short_answer"]
            row_result["answer_pos"] = meta["answer_pos"]
            row_result["query"] = meta["query"]

            if thinking_content:
                row_result["thinking_content"] = thinking_content

            results.append(row_result)

    return results

## Utils

In [13]:
def export_jsonl(results: List[Dict], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None

    allowed_fields = [
        "query_id",
        "retrieved_chunks",
        "augmented_prompt",
        "generated_answer",
    ]

    with open(path, "w", encoding="utf-8") as f:
        for result in results:
            item = {
                field: result[field]
                for field in allowed_fields
                if field in result
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def make_submission_path(
    output_dir: str,
    split: str,
    variant: str,
) -> str:
    filename = f"doubleN-{split}-{variant}.jsonl"
    return os.path.join(output_dir, filename)

In [14]:
def display_results_table(results: List[Dict], n: int = 5):
    preview_df = pd.DataFrame(results[:n])

    cols = [
        "query_id",
        "query",
        "retrieved_chunks",
        "generated_answer",
        "short_answer",
    ]

    cols = [c for c in cols if c in preview_df.columns]

    display(
        preview_df[cols].style.set_properties(
            **{
                "white-space": "pre-wrap",
                "text-align": "left",
                "vertical-align": "top",
            }
        )
    )

# Experiments

### Get drive + files

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
data_module = DataModule(dev_size=0.2)
ds = data_module.load(ds_url)

train_df, val_df = data_module.build_train_val_split()
test_df = data_module.to_dataframe("test")
blind_df = data_module.to_dataframe("blind")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

data/blind-00000-of-00001.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating blind split:   0%|          | 0/1322 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [17]:
retrieval_store = RetrievalStore()
prompt_builder = PromptBuilder()

In [18]:
baseline_test_df = retrieval_store.attach_rankings_from_jsonl(
    test_df,
    test_ranking_path,
)

baseline_blind_df = retrieval_store.attach_rankings_from_jsonl(
    blind_df,
    blind_ranking_path,
)

## [B1]: Implement at least two small LMs (<= 3B parameters) zero-shot in (a) baseline, (b) RAG and (c) Oracle settings

In [19]:
runner = ExperimentRunner(
    prompt_builder,
    retrieval_store,
)

### Qwen3-0.6B

In [21]:
qwen3_06B_model = QALLM(model_name="Qwen/Qwen3-0.6B")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Baseline setting

In [ ]:
results = runner.retrieve_answers(
    test_df,
    qwen3_06B_model,
    setting="baseline",
)

display_results_table(results, n=50)

In [ ]:
export_jsonl(
    results,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-Qwen3-0.6B-b1-baseline.jsonl",
)

In [ ]:
AnswerEvaluator.evaluate_dataset(qwen3_06B_rag_base_test)

RAG setting

In [ ]:
qwen3_06B_rag_base_test = runner.retrieve_answers(
    df=baseline_test_df,
    model=qwen3_06B_model,
    setting="rag",
)

export_jsonl(
    qwen3_06B_rag_base_test,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-Qwen3-0.6B-b1-rag.jsonl",
)

rag generation:   0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
results = AnswerEvaluator.evaluate_dataset(qwen3_06B_rag_base_test)

In [ ]:
results

{'EM': 0.0005, 'subEM': 0.452, 'METEOR': 0.304642752661458}

In [ ]:
qwen3_06B_rag_base_blind = runner.retrieve_answers(
    df=baseline_blind_df,
    model=qwen3_06B_model,
    setting="rag",
)

export_jsonl(
    qwen3_06B_rag_base_blind,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-blind-Qwen3-0.6B-b1-rag.jsonl",
)

Oracle setting

In [ ]:
qwen3_06B_orac_base_test = runner.retrieve_answers(
    df=baseline_test_df,
    model=qwen3_06B_model,
    setting="oracle",
)



In [24]:
export_jsonl(
    qwen3_06B_orac_base_test,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-Qwen3-0.6B-b1-oracle.jsonl",
)

In [25]:
display_results_table(qwen3_06B_orac_base_test, n=5)


,query_id,query,retrieved_chunks,generated_answer,short_answer
0,4HZKjht3X13n,where is the lady with an ermine located,"[0, 10, 4]",The lady with an ermine is located in the main building of the National Museum in Kraków,['the National Museum in Kraków']
1,BeCayzNq9ZgD,lok sabha or rajya sabha which is bigger,"[12, 11, 18]",The Lok Sabha is **larger** than the Rajya Sabha. Here's why:,['Lok Sabha']
2,ZP7FeO0UVt3U,who was the actress who played pinky tuscadero on happy days,"[0, 2, 3]",The actress who played Pinky Tuscadero on Happy Days is **Roz Kelly**,['Roz Kelly']
3,Tt88r7IhbHtg,who wrote the song i want you back,"[2, 0, 1]","The song ""I Want You Back"" was written by **Michael Jackson**",['Berry GordyFreddie PerrenAlphonso MizellDeke Richards']
4,PblIzyQ705Kg,who wrote the music for fiddler on the roof,"[0, 2, 33]",The music for *Fiddler on the Roof* was written by **Rodgers and Hammerstein**,['Jerry Bock']


In [27]:
AnswerEvaluator.evaluate_dataset(qwen3_06B_orac_base_test)

{'EM': 0.0, 'subEM': 0.487, 'METEOR': 0.3113308016875837}

### SmolLM2-1.7B

In [29]:
smollm2_1_7B_model = SmolLM()

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Baseline setting

In [32]:
smollm2_1_7B_base_test = runner.retrieve_answers(
    test_df,
    smollm2_1_7B_model,
    setting="baseline",
)

display_results_table(smollm2_1_7B_base_test, n=5)

baseline generation:   0%|          | 0/250 [00:00<?, ?it/s]

,query_id,query,retrieved_chunks,generated_answer,short_answer
0,4HZKjht3X13n,where is the lady with an ermine located,[],"The Lady with an ermine is located in the painting ""The Lady with an Ermine"" by Leonardo da Vinci",['the National Museum in Kraków']
1,BeCayzNq9ZgD,lok sabha or rajya sabha which is bigger,[],Rajya Sabha,['Lok Sabha']
2,ZP7FeO0UVt3U,who was the actress who played pinky tuscadero on happy days,[],Tina Louise Gold,['Roz Kelly']
3,Tt88r7IhbHtg,who wrote the song i want you back,[],"""I Want You Back"" was written by John Fogerty and Tom Scott",['Berry GordyFreddie PerrenAlphonso MizellDeke Richards']
4,PblIzyQ705Kg,who wrote the music for fiddler on the roof,[],Anatole Chuikov,['Jerry Bock']


In [33]:
export_jsonl(
    smollm2_1_7B_base_test,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-smollm2-7B-b1-baseline.jsonl",
)

In [34]:
AnswerEvaluator.evaluate_dataset(smollm2_1_7B_base_test)

{'EM': 0.0435, 'subEM': 0.1335, 'METEOR': 0.12556485309520612}

RAG setting

In [ ]:
smollm2_1_7B_rag_base_test = runner.retrieve_answers(
    df=test_df,
    model=smollm2_1_7B_model,
    setting="rag",
)

export_jsonl(
    smollm2_1_7B_rag_base_test,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-test-SmolLM2-1.7B-b1-rag.jsonl",
)

In [ ]:
smollm2_1_7B_rag_base_blind = runner.retrieve_answers(
    df=test_df,
    model=smollm2_1_7B_model,
    setting="rag",
)

export_jsonl(
    smollm2_1_7B_rag_base_blind,
    f"/content/drive/MyDrive/{output_dir}/outputs/doubleN-all-blind-SmolLM2-1.7B-b1-rag.jsonl",
)

Oracle setting

In [ ]:
smollm2_1_7B_orac_base_test = runner.retrieve_answers(
    df=test_df,
    model=smollm2_1_7B_model,
    setting="oracle",
)

display_results_table(smollm2_1_7B_orac_base_test, n=5)

## [B2]: Evaluation Framework